# Session-key protection on the sensor → QSES link

**The problem:** in the paper's protocol the sensor sends its reading to the QSES **unencrypted**, so anyone on that link can read it.

**The fix:** the sensor and the QSES run one Kyber (ML-KEM-512) handshake per session to agree on an AES key. Every reading after that is encrypted with AES-GCM. Kyber is heavy, so it runs once per session, not once per reading.

**This notebook:**
1. Setup
2. The handshake, step by step
3. Before and after: what an eavesdropper sees
4. The reading still reaches the server
5. Attacks on the new link

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import struct

from src.entities.client import ClientNode                  # the paper's sensor (sends plaintext)
from src.entities.secure_client import SecureClientNode     # new: sensor with a session key
from src.entities.secure_qses import SecureQSESNode         # new: QSES that answers the handshake
from src.entities.server import MedicalServerNode
from src.protocol.errors import ProtocolError
from src.protocol.secure_messages import LINK_PACKET_SIZE, SensorLinkPacket

print("Imports OK, project root:", project_root)

liboqs-python faulthandler is disabled
Imports OK, project root: /home/girdh/coding/experiments/pq-iomt-framework


## 1. Setup

`make_system()` creates the three machines and gives each one the keys it needs:

| Line | What it does |
|---|---|
| `qses.register_client(1, client.public_key)` | The QSES learns sensor 1's **Dilithium public key**, so it can check that sensor's signatures |
| `server.trust_qses(qses.public_key)` | The server learns the QSES's **Dilithium public key**, so it can check the QSES's signature on each packet |
| `server.register_client(1, client.public_key, client.hmac_key, 222)` | The server learns sensor 1's public key, its **HMAC key**, and the QSES nonce to expect. The QSES never gets the HMAC key |

`handshake()` runs the whole handshake in one call (the next section does it step by step).
`expect_rejected()` runs an attack and prints whether the system rejected it.

The QSES freshness window is set to 10 minutes so you can run cells by hand without hitting `stale timestamp`.

In [2]:
def make_system():
    server = MedicalServerNode(server_id=9)
    qses = SecureQSESNode(server.public_key, max_age_ms=10 * 60 * 1000)
    client = SecureClientNode(client_id=1, server_id=9, session_id=3, qses_nonce=222,
                              qses_public_key=qses.public_key)
    qses.register_client(1, client.public_key)
    server.trust_qses(qses.public_key)
    server.register_client(1, client.public_key, client.hmac_key, 222)
    return server, qses, client


def handshake(qses, client):
    cid, n_i = client.begin_session()
    pk_kem, n_q, sid, sig_q = qses.offer_session(cid, n_i)
    ct, sig_d = client.complete_session(pk_kem, n_q, sid, sig_q)
    return qses.finish_session(cid, ct, sig_d)


def expect_rejected(action):
    try:
        action()
        print("!! NOT rejected (this would be a bug)")
    except ProtocolError as e:
        print(f"[REJECTED] {e}")

print("helpers defined")

helpers defined


## 2. The handshake (once per session)

Think of it as a short conversation:

| Step | Who | What happens |
|---|---|---|
| 1 | Sensor → QSES | "I'm sensor 1. Here is a random number `n_i`." |
| 2 | QSES → Sensor | **The offer:** "Here is my Kyber padlock `pk_kem`, a session label `session_id`, my own random number `n_q`, and my signature over all of it, so you know it's really me." |
| 3 | Sensor | Checks the signature. Then **Encaps**: locks a secret `ss` in a box `ct` using the padlock |
| 4 | Sensor → QSES | "Here is the box `ct`, and my signature so you know it's me." |
| 5 | QSES | Checks the signature. Then **Decaps**: opens the box with its private key and gets the same `ss` |
| 6 | Both | `ss` goes through HKDF to become the AES key `K` |

The random numbers (`n_i`, `n_q`) stop an attacker replaying an old recorded handshake. The signatures stop a fake QSES or a fake sensor.

In [3]:
server, qses, client = make_system()

# Step 1: sensor -> QSES
cid, n_i = client.begin_session()
print("1. sensor -> QSES :  id =", cid, "| n_i =", n_i.hex()[:8], "...")

# Step 2: QSES -> sensor (the offer)
pk_kem, n_q, sid, sig_q = qses.offer_session(cid, n_i)
print("2. QSES -> sensor :  pk_kem", len(pk_kem), "B | n_q", n_q.hex()[:8], "... | session_id", sid, "| signature", len(sig_q), "B")

# Steps 3-4: sensor checks the offer, runs Encaps, signs the reply
ct, sig_d = client.complete_session(pk_kem, n_q, sid, sig_q)
print("3. sensor verified the QSES signature and ran Encaps")
print("4. sensor -> QSES :  ct", len(ct), "B | signature", len(sig_d), "B")

# Step 5: QSES checks the reply, runs Decaps
qses.finish_session(cid, ct, sig_d)
print("5. QSES verified the sensor signature and ran Decaps")

# Step 6: both sides now hold the same key
same_key = client._link_cipher.key == qses.sessions[sid].key
print("6. both sides hold the same AES key:", same_key)
assert same_key

1. sensor -> QSES :  id = 1 | n_i = 0f7510ba ...
2. QSES -> sensor :  pk_kem 800 B | n_q 44be2d91 ... | session_id 1522966513 | signature 2420 B
3. sensor verified the QSES signature and ran Encaps
4. sensor -> QSES :  ct 768 B | signature 2420 B
5. QSES verified the sensor signature and ran Decaps
6. both sides hold the same AES key: True


## 3. Before and after: what an eavesdropper sees

An eavesdropper copies the raw bytes off the sensor → QSES link. It has no keys.

### Before (the paper's version)

In [4]:
baseline_sensor = ClientNode(client_id=2, server_id=9, session_id=3, qses_nonce=222)
plain = baseline_sensor.create_signed_reading(b"Heart rate: 78").to_bytes()

print("raw bytes:", plain[:40])

reading = plain[:100].rstrip(b"\x00").decode()
client_id, seq, t_ms, _, _, session, server_id = struct.unpack(">IIQIIII", plain[100:132])
print("reading :", reading)
print("device  :", client_id, "| seq:", seq, "| time(ms):", t_ms)
print("session :", session, "| destination server:", server_id)

raw bytes: b'Heart rate: 78\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'
reading : Heart rate: 78
device  : 2 | seq: 1 | time(ms): 1791994200800
session : 3 | destination server: 9


### After (session key)

In [5]:
link_pkt = client.create_encrypted_reading(b"Heart rate: 78")
captured = link_pkt.to_bytes()

print("raw bytes:", captured[:40])
print("reading visible:", b"Heart rate" in captured)

assert len(captured) == LINK_PACKET_SIZE == 2616
assert b"Heart rate" not in captured
print("\nConfirmed: the eavesdropper sees only ciphertext on the sensor -> QSES link.")

raw bytes: b'Z\xc6\x9f\xf1\xfe\x84d\x80\x1b\x0f\x02\xdd\xda\x05\xe3b\x81\x0bi?\x84\xf87\xdc\xf9C\x92\xf0\xfc\rpf\xf0\x15\x9b\x02\xaa\xa2\xc0\xb7'
reading visible: False

Confirmed: the eavesdropper sees only ciphertext on the sensor -> QSES link.


## 4. The reading still reaches the server

The QSES decrypts the sensor's packet, then runs the paper's Algorithm 2 unchanged (freshness, replay check, sensor signature, then Kyber + AES to the server). The server verifies and stores the reading as before.

In [6]:
edge_packet = qses.receive_from_sensor(link_pkt)
stored = server.receive_and_verify(edge_packet)

print("server stored:", stored.message.rstrip(b"\x00"))
print("client_id:", stored.client_id, "| seq:", stored.seq)

server stored: b'Heart rate: 78'
client_id: 1 | seq: 1


### What the fix costs

In [7]:
print("Packet sensor -> QSES, per reading")
print("  paper   :", len(plain), "bytes")
print("  session :", len(captured), "bytes  (+", len(captured) - len(plain), ": session id, IV, GCM tag)")

print("\nHandshake, once per session")
print("  QSES -> sensor :", len(pk_kem) + len(n_q) + 4 + len(sig_q), "bytes")
print("  sensor -> QSES :", len(ct) + len(sig_d), "bytes")

Packet sensor -> QSES, per reading
  paper   : 2584 bytes
  session : 2616 bytes  (+ 32 : session id, IV, GCM tag)

Handshake, once per session
  QSES -> sensor : 3240 bytes
  sensor -> QSES : 3188 bytes


## 5. Attacks on the new link

Each attack gets a fresh system, so the cells do not affect each other.

| Attack | What the attacker does | Which check stops it |
|---|---|---|
| A | Flips one bit of the ciphertext | AES-GCM tag |
| B | Changes the session id in the header | Unknown session |
| C | Replays a captured packet | Sequence number check |
| D | Sets up a rogue box pretending to be the QSES | Sensor checks the QSES signature |
| E | Replays an old recorded offer | `n_i` is inside the signed offer |
| F | Sends a forged handshake reply as the sensor | QSES checks the sensor signature |
| G | n/a (sanity check) | Sensor refuses to send without a session |

### Attack A: flip one bit in the ciphertext

In [8]:
s, q, c = make_system()
handshake(q, c)

pkt = c.create_encrypted_reading(b"Heart rate: 78")
bad_c = bytes([pkt.ciphertext[0] ^ 1]) + pkt.ciphertext[1:]
tampered = SensorLinkPacket(pkt.session_id, pkt.iv, bad_c, pkt.tag)

expect_rejected(lambda: q.receive_from_sensor(tampered))

[REJECTED] sensor link: AES-GCM authentication failed


### Attack B: change the session id in the header

In [9]:
s, q, c = make_system()
handshake(q, c)

pkt = c.create_encrypted_reading(b"Heart rate: 78")
wrong_session = SensorLinkPacket(pkt.session_id ^ 1, pkt.iv, pkt.ciphertext, pkt.tag)

expect_rejected(lambda: q.receive_from_sensor(wrong_session))

[REJECTED] unknown session


### Attack C: replay a captured packet

In [10]:
s, q, c = make_system()
handshake(q, c)

pkt = c.create_encrypted_reading(b"Heart rate: 78")
q.receive_from_sensor(pkt)
print("first delivery accepted, now replaying the same packet...")

expect_rejected(lambda: q.receive_from_sensor(pkt))

first delivery accepted, now replaying the same packet...
[REJECTED] replayed or out-of-order sequence number


### Attack D: a rogue box pretends to be the QSES

The rogue box has its own signing key, so the sensor cannot verify its offer against the real `pk_QSES`.

In [11]:
s, q, c = make_system()
rogue = SecureQSESNode(s.public_key)
rogue.register_client(1, c.public_key)

cid, n_i = c.begin_session()
rogue_offer = rogue.offer_session(cid, n_i)

expect_rejected(lambda: c.complete_session(*rogue_offer))

[REJECTED] invalid QSES offer signature


### Attack E: replay an old recorded offer

The offer for today's request is signed over today's `n_i`. A recording from an earlier request carries an old `n_i`.

In [12]:
s, q, c = make_system()

cid, n_i = c.begin_session()
old_offer = q.offer_session(cid, n_i)      # the attacker records this
c.begin_session()                          # later: a new request, new n_i

expect_rejected(lambda: c.complete_session(*old_offer))

[REJECTED] invalid QSES offer signature


### Attack F: a forged handshake reply

In [13]:
s, q, c = make_system()

cid, n_i = c.begin_session()
pk, nq, session, sq = q.offer_session(cid, n_i)
box, real_sig = c.complete_session(pk, nq, session, sq)

forged_sig = b"\x00" * len(real_sig)
expect_rejected(lambda: q.finish_session(cid, box, forged_sig))

[REJECTED] invalid handshake signature


### Check G: no session, no sending

In [14]:
s, q, c = make_system()      # handshake NOT run
expect_rejected(lambda: c.create_encrypted_reading(b"Heart rate: 78"))

[REJECTED] no session: run the handshake first


## Summary

- **Fixed:** passive eavesdropping on the sensor → QSES link. Before, the reading, device id, timestamps and destination were readable. After, the eavesdropper sees ciphertext.
- **Cost:** about 32 extra bytes per reading, plus one handshake per session.
- **Preserved:** the sensor's signature still travels to the server, so end-to-end authentication is unchanged.

**Limits to state:**
- The QSES still decrypts the reading (it must, to verify the signature). This protects the link from outsiders, not from the QSES itself.
- The QSES → server hop still uses the server's long-term Kyber key.
- The `session_id` and `n_q` inside `meta` are still the provisioned values, because the server is unchanged.
- Timings come from a normal computer, not a real sensor.